# JPMC Consumer Banking - Time Travel & Zero Copy Clone

This notebook demonstrates two powerful Snowflake features:

1. **Time Travel** — Query historical data as it existed at any point within the retention period
2. **Zero Copy Clone** — Instantly create full copies of tables, schemas, or databases without duplicating storage

**Database:** `JPMC_CONSUMER_BANKING_HOL.HOL_LAB`  
**Warehouse:** `HOL_MEDIUM_WH`  
**Tables:** ACCOUNTS, LOANS, CREDIT_CARDS

In [ ]:
-- =============================================================================
-- SETUP
-- =============================================================================

USE ROLE ACCOUNTADMIN;
USE WAREHOUSE HOL_MEDIUM_WH;
USE SCHEMA JPMC_CONSUMER_BANKING_HOL.HOL_LAB;

## Part 1: Time Travel

Snowflake automatically retains historical versions of data for a configurable retention period (up to 90 days on Enterprise+). This enables:
- Querying data as of a past timestamp (`AT`)
- Querying data before a specific change (`BEFORE`)
- Restoring accidentally deleted or modified data
- Auditing changes over time

In [ ]:
-- =============================================================================
-- Step 1: Check current state and retention settings
-- =============================================================================

-- View current row count and data retention setting
SELECT COUNT(*) AS current_row_count FROM CUSTOMERS;

SHOW PARAMETERS LIKE 'DATA_RETENTION_TIME_IN_DAYS' IN TABLE CUSTOMERS;

In [ ]:
-- =============================================================================
-- Step 2: Make a change — simulate an accidental UPDATE
-- =============================================================================

-- Record the current timestamp before making changes
SET ts_before_update = CURRENT_TIMESTAMP();

-- Check current segments for active customers
SELECT CUSTOMER_ID, FIRST_NAME, LAST_NAME, SEGMENT, RISK_RATING, IS_ACTIVE
FROM CUSTOMERS
WHERE IS_ACTIVE = TRUE
LIMIT 5;

-- Simulate accidental update: downgrade all active customers to 'Standard' segment
UPDATE CUSTOMERS
SET SEGMENT = 'Standard'
WHERE IS_ACTIVE = TRUE;

SELECT 'Rows updated' AS action, COUNT(*) AS affected_rows
FROM CUSTOMERS WHERE IS_ACTIVE = TRUE AND SEGMENT = 'Standard';

In [ ]:
-- =============================================================================
-- Step 3: Use Time Travel to see data BEFORE the accident
-- =============================================================================

-- Query the table as it was before the update
SELECT CUSTOMER_ID, FIRST_NAME, LAST_NAME, SEGMENT, RISK_RATING, IS_ACTIVE
FROM CUSTOMERS AT(TIMESTAMP => $ts_before_update)
WHERE IS_ACTIVE = TRUE
LIMIT 5;

-- Compare current vs historical: how many customers lost their premium segments?
SELECT
    'Before Update' AS state,
    COUNT(*) AS active_customers,
    COUNT(CASE WHEN SEGMENT = 'Premium' THEN 1 END) AS premium_count,
    COUNT(CASE WHEN SEGMENT = 'Gold' THEN 1 END) AS gold_count
FROM CUSTOMERS AT(TIMESTAMP => $ts_before_update)
WHERE IS_ACTIVE = TRUE
UNION ALL
SELECT
    'After Update' AS state,
    COUNT(*) AS active_customers,
    COUNT(CASE WHEN SEGMENT = 'Premium' THEN 1 END) AS premium_count,
    COUNT(CASE WHEN SEGMENT = 'Gold' THEN 1 END) AS gold_count
FROM CUSTOMERS
WHERE IS_ACTIVE = TRUE;

In [ ]:
-- =============================================================================
-- Step 4: Restore the data using Time Travel
-- =============================================================================

-- Restore using CREATE OR REPLACE from time travel
CREATE OR REPLACE TABLE CUSTOMERS AS
SELECT * FROM CUSTOMERS AT(TIMESTAMP => $ts_before_update);

-- Verify restoration — Premium and Gold customers should be back
SELECT
    COUNT(*) AS total_rows,
    COUNT(CASE WHEN SEGMENT = 'Standard' AND IS_ACTIVE = TRUE THEN 1 END) AS standard_active,
    COUNT(CASE WHEN SEGMENT = 'Premium' THEN 1 END) AS premium_count,
    COUNT(CASE WHEN SEGMENT = 'Gold' THEN 1 END) AS gold_count
FROM CUSTOMERS;

In [ ]:
-- =============================================================================
-- Step 5: Time Travel with STATEMENT ID
-- =============================================================================

-- Query using a specific statement ID (useful for undoing a known query)
-- BEFORE keyword gives data just before that statement executed
-- SELECT * FROM CUSTOMERS BEFORE(STATEMENT => '<query_id>');

In [ ]:
-- =============================================================================
-- Step 6: Time Travel on DROPPED tables (UNDROP)
-- =============================================================================

-- Drop a table
DROP TABLE CUSTOMERS;

-- Oops! Restore it with UNDROP
UNDROP TABLE CUSTOMERS;

-- Verify it's back
SELECT COUNT(*) AS restored_rows FROM CUSTOMERS;

## Part 2: Zero Copy Clone

Zero Copy Clone creates an instant, metadata-only copy of a table, schema, or database. Key properties:
- **No additional storage cost** at creation time (shares underlying micro-partitions)
- **Independent after creation** — changes to clone don't affect source and vice versa
- **Storage only grows** when cloned data diverges (copy-on-write)
- **Supports** tables, schemas, databases, stages, file formats, sequences, streams, and tasks

In [ ]:
-- =============================================================================
-- Step 7: Clone a table (instant, zero storage)
-- =============================================================================

-- Clone the CUSTOMERS table
CREATE OR REPLACE TABLE CUSTOMERS_CLONE CLONE CUSTOMERS;

-- Verify: same data, zero additional storage
SELECT
    'CUSTOMERS' AS table_name, COUNT(*) AS row_count FROM JPMC_CONSUMER_BANKING_HOL.HOL_LAB.CUSTOMERS
UNION ALL
SELECT
    'CUSTOMERS_CLONE', COUNT(*) FROM JPMC_CONSUMER_BANKING_HOL.HOL_LAB.CUSTOMERS_CLONE;

-- Check storage — clone initially uses 0 bytes
SELECT TABLE_NAME,
       SUM(ACTIVE_BYTES) AS ACTIVE_BYTES,
       SUM(TIME_TRAVEL_BYTES) AS TIME_TRAVEL_BYTES,
       SUM(RETAINED_FOR_CLONE_BYTES) AS RETAINED_FOR_CLONE_BYTES
FROM JPMC_CONSUMER_BANKING_HOL.INFORMATION_SCHEMA.TABLE_STORAGE_METRICS
WHERE TABLE_NAME IN ('CUSTOMERS', 'CUSTOMERS_CLONE')
AND TABLE_CATALOG = 'JPMC_CONSUMER_BANKING_HOL'
AND TABLE_SCHEMA = 'HOL_LAB'
GROUP BY TABLE_NAME;

In [ ]:
-- =============================================================================
-- Step 8: Modify the clone (independent from source)
-- =============================================================================

-- Delete inactive customers from the clone — source table is NOT affected
DELETE FROM CUSTOMERS_CLONE WHERE IS_ACTIVE = FALSE;

-- Add a derived column to the clone
ALTER TABLE CUSTOMERS_CLONE ADD COLUMN RISK_TIER VARCHAR;

UPDATE CUSTOMERS_CLONE
SET RISK_TIER = CASE
    WHEN RISK_RATING = 'Low' AND SEGMENT IN ('Premium', 'Gold') THEN 'LOW_RISK'
    WHEN RISK_RATING = 'Medium' THEN 'MODERATE_RISK'
    ELSE 'HIGH_RISK'
END;

-- Compare: source unchanged, clone modified
SELECT 'SOURCE' AS table_type, COUNT(*) AS row_count, NULL AS has_risk_tier
FROM CUSTOMERS
UNION ALL
SELECT 'CLONE', COUNT(*), 'YES'
FROM CUSTOMERS_CLONE;

In [ ]:
-- =============================================================================
-- Step 9: Clone an entire SCHEMA (all tables at once)
-- =============================================================================

-- Clone the full schema for dev/testing — instant, no storage duplication
CREATE OR REPLACE SCHEMA HOL_LAB_DEV CLONE HOL_LAB;

-- Verify all tables were cloned
SHOW TABLES IN SCHEMA JPMC_CONSUMER_BANKING_HOL.HOL_LAB_DEV;

-- Query from the cloned schema
SELECT COUNT(*) AS dev_customers FROM JPMC_CONSUMER_BANKING_HOL.HOL_LAB_DEV.CUSTOMERS;

In [ ]:
-- =============================================================================
-- Step 11: Use Cases Summary
-- =============================================================================

-- Real-world use cases demonstrated:
--
-- TIME TRAVEL:
--   1. Undo accidental DML (UPDATE/DELETE) without backup restores
--   2. Audit data changes over time
--   3. Recover dropped tables (UNDROP)
--   4. Point-in-time reporting
--
-- ZERO COPY CLONE:
--   1. Instant dev/test environments from production data
--   2. Safe experimentation without affecting source
--   3. Point-in-time snapshots (clone + time travel)
--   4. Data sandboxes for analytics teams
--   5. Pre-release testing of schema changes

SELECT 'Time Travel + Zero Copy Clone Demo Complete' AS status;

In [ ]:
-- =============================================================================
-- CLEANUP (optional)
-- =============================================================================

DROP TABLE IF EXISTS CUSTOMERS_CLONE;
DROP SCHEMA IF EXISTS JPMC_CONSUMER_BANKING_HOL.HOL_LAB_DEV;

SELECT 'Cleanup complete' AS status;